In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/X_feat_pruned.pkl
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__results__.html
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/train_idx.npy
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/test_idx.npy
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__notebook__.ipynb
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/class_weights.pkl
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__output__.json
/kaggle/input/notebooks/arijitmohanta/03-class-imbalance/custom.css
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results__.html
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_feat.pkl
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/y.npy
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_img.npy
/kaggle/input/notebooks/arijitmohanta/02-preprocessing/__output__.json
/kaggle/i

In [3]:
# ============================================================
# STAGE 4.1 — LOCATE THE STAGE 2 + STAGE 3 ARTIFACTS ON KAGGLE
# Stage 4 draws files from BOTH previous notebooks:
#       - from 02-preprocessing  : X_img.npy, y.npy          (raw images + labels),
#       - from 03-class-imbalance: X_feat_pruned.pkl,        (15-col feature table),
#                                  train_idx.npy, test_idx.npy (the stratified split),
#                                  class_weights.pkl          (balanced weights).
# Two inputs = two mount folders, each with its own slug, so we
#       1. Walk the whole input tree and print every file with its size,
#       2. Resolve the six files we need into full paths,
#       3. Surface any that are missing before Step 2 tries to load them.
# ============================================================

import os                                     # stdlib; all that's needed to walk the mounts

# Walk every folder under /kaggle/input and print each file with its size in MB.
# The listing should show two folders — the big image tensor coming from Stage 2
# The small split/weight files from Stage 3 — which visually confirms both inputs attached, not just one.
print("Full listing under /kaggle/input:\n")
for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        full = os.path.join(dirpath, f)
        print(f"{os.path.getsize(full) / 1e6:8.1f} MB   {full}")

# Resolve the six files Step 2 loads. 
# Walking rather than hardcoding keeps this robust to Kaggle's slug-based mount paths, exactly as in Stages 2 and 3.
targets = ['X_img.npy', 'y.npy',                       # from 02-preprocessing
           'X_feat_pruned.pkl', 'train_idx.npy',       # from 03-class-imbalance
           'test_idx.npy', 'class_weights.pkl']
found = {}
for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        if f in targets:                               # exact filename match only
            found[f] = os.path.join(dirpath, f)

print("\nTargets found:")
for t in targets:
    print(f"  {t:20s} {found.get(t, 'NOT FOUND')}")    # 'NOT FOUND' = missing/mis-attached input

Full listing under /kaggle/input:

    20.8 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/X_feat_pruned.pkl
     0.3 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__results__.html
     1.1 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/train_idx.npy
     0.3 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/test_idx.npy
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__notebook__.ipynb
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/class_weights.pkl
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/__output__.json
     0.0 MB   /kaggle/input/notebooks/arijitmohanta/03-class-imbalance/custom.css
     0.4 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__results__.html
    22.1 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/X_feat.pkl
     0.2 MB   /kaggle/input/notebooks/arijitmohanta/02-preprocessing/__notebook__.ipynb
     6.2 MB   /ka

In [4]:
# ============================================================
# STAGE 4.2 — LOAD ARTIFACTS + REBUILD THE FROZEN SPLIT
# Bring in the six files and reconstruct the train/test partition by
# APPLYING the saved indices, never re-splitting. This cell:
#       1. loads images, features, labels, split indices, weights,
#       2. slices each array with train_idx / test_idx,
#       3. asserts the split is disjoint, complete, and class-correct.
# Applying frozen indices is what keeps the classical and CNN routes on the identical wafers — the precondition for comparing them fairly.
# ============================================================

import numpy as np
import pandas as pd
import pickle

# Resolve the two mount folders once, so the loads below read from the confirmed
# Step-1 paths and there's a single place to edit if a slug ever changes.
P2 = '/kaggle/input/notebooks/arijitmohanta/02-preprocessing'
P3 = '/kaggle/input/notebooks/arijitmohanta/03-class-imbalance'

# Load the raw artifacts. X_img stays uint8 (categorical pixels); X_feat carries
# its 15 named columns; y is the shared label array; the two idx arrays are the
# frozen split; class_weights is the {label: weight} dict from Stage 3.
X_img  = np.load(f'{P2}/X_img.npy')                       # (172950, 64, 64) uint8
y      = np.load(f'{P2}/y.npy')                           # (172950,) <U9 strings
X_feat = pd.read_pickle(f'{P3}/X_feat_pruned.pkl')        # (172950, 15) named table
train_idx = np.load(f'{P3}/train_idx.npy')                # 138360 train row positions
test_idx  = np.load(f'{P3}/test_idx.npy')                 # 34590 test row positions
with open(f'{P3}/class_weights.pkl', 'rb') as f:
    class_weights = pickle.load(f)                        # {label: balanced weight}

# Apply the frozen indices. 
# Feature route uses .iloc (DataFrame), image route uses fancy-indexing (ndarray);
# both preserve order, so row i is the same wafer across X_feat, X_img, and y within each partition.
X_feat_train, X_feat_test = X_feat.iloc[train_idx], X_feat.iloc[test_idx]
X_img_train,  X_img_test  = X_img[train_idx],       X_img[test_idx]
y_train,      y_test      = y[train_idx],           y[test_idx]

# --- Integrity checks on the reconstructed split ---
# 1. Disjoint + complete: no wafer is on both sides, and together they cover all N.
assert len(set(train_idx) & set(test_idx)) == 0, "Train/test indices overlap — leakage!"
assert len(train_idx) + len(test_idx) == len(y), "Split doesn't cover all rows!"
# 2. All 9 classes are present on both sides (guards macro-F1 being computable in Stage 5).
assert set(y_train) == set(y_test) == set(y), "A class is missing from one side!"

print("Feature route :", X_feat_train.shape, "train /", X_feat_test.shape, "test")
print("Image route   :", X_img_train.shape,  "train /", X_img_test.shape,  "test")
print("Classes on both sides:", len(set(y_train)))
print("\nSplit reconstructed and verified — safe to model.")

Feature route : (138360, 15) train / (34590, 15) test
Image route   : (138360, 64, 64) train / (34590, 64, 64) test
Classes on both sides: 9

Split reconstructed and verified — safe to model.


In [5]:
# ============================================================
# STAGE 4.3 — RANDOM FOREST BASELINE (classical route, interpretable anchor)
# The first model. 
# Random Forest is chosen to lead because its feature-importance output is directly readable in process terms.
# This cell:
#       1. trains an RF using the Stage 3 balanced class weights,
#       2. reports macro-F1 + accuracy (the gap between them is the point),
#       3. prints the per-class precision/recall/F1 table,
#       4. Ranks the 15 hand features by importance.
# CPU only; no GPU needed for the classical route.
# ============================================================

import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score

# Order the classes by training frequency (none first, Near-full last)
# Every report below reads in the same familiar order as the Stage 3 tables, rather than sklearn's default alphabetical shuffle.
class_order = pd.Series(y_train).value_counts().index.tolist()

# Train the forest. 
# class_weight receives the Stage 3 dict directly
# a Donut error costs ~35x and a Near-full error ~129x a 'none' error inside the split criterion. 
# n_estimators=300 is a solid baseline depth/variance trade; 
# n_jobs=-1 uses all Kaggle cores; random_state fixes the bootstrap + feature sampling
# The run reproduces exactly.
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight=class_weights,     # the audited Stage 3 weights, applied at split time
    random_state=42,
    n_jobs=-1                       # parallelize across all available CPU cores
)
t0 = time.time()
rf.fit(X_feat_train, y_train)
print(f"Trained in {time.time() - t0:.1f}s")

# Predict on the held-out test wafers (real ~989:1 distribution, untouched).
y_pred = rf.predict(X_feat_test)

# Headline metrics. 
# We print accuracy AND macro-F1 side by side precisely to make the imbalance trap visible: 
# accuracy is inflated by the 85% 'none' majority,
# while macro-F1 averages all nine classes equally and is the honest score.
print(f"\nAccuracy : {accuracy_score(y_test, y_pred):.4f}   <- inflated by the 'none' majority")
print(f"Macro-F1 : {f1_score(y_test, y_pred, average='macro'):.4f}   <- the metric we steer by")

# Full per-class report: precision / recall / F1 / support for every class, in the
# frequency order set above. This is where the rare classes reveal themselves.
print("\nPer-class report:")
print(classification_report(y_test, y_pred, labels=class_order, digits=3))

# Rank the hand features by the forest's impurity-based importance. 
# This ordering is the interpretable payload of the classical route: 
# it says which physical signal (radial non-uniformity, collinearity, connectivity, angular spread) carried the classification. 
# Impurity importance can favor continuous features,
# so we read it as a strong hint, to be confirmed, not gospel.
importances = pd.Series(rf.feature_importances_, index=X_feat.columns).sort_values(ascending=False)
print("\nFeature importances (impurity-based, descending):")
print(importances.round(4))

Trained in 55.5s

Accuracy : 0.9632   <- inflated by the 'none' majority
Macro-F1 : 0.7893   <- the metric we steer by

Per-class report:
              precision    recall  f1-score   support

        none      0.972     0.997     0.984     29486
   Edge-Ring      0.981     0.941     0.960      1936
    Edge-Loc      0.776     0.684     0.727      1038
      Center      0.952     0.860     0.904       859
         Loc      0.787     0.490     0.604       718
     Scratch      0.770     0.197     0.313       239
      Random      0.894     0.832     0.862       173
       Donut      0.822     0.748     0.783       111
   Near-full      1.000     0.933     0.966        30

    accuracy                          0.963     34590
   macro avg      0.884     0.742     0.789     34590
weighted avg      0.960     0.963     0.959     34590


Feature importances (impurity-based, descending):
conn_fragmentation    0.0952
radial_ring_1         0.0938
conn_largest_frac     0.0928
radial_ring_8      

In [6]:
# ============================================================
# STAGE 4.4 — XGBOOST (classical route, gradient-boosted comparison)
# Second model, held to the exact same split and metrics as the RF
# The comparison is clean. 
# Two XGBoost-specific mechanics are handled here:
#       (a) imbalance is applied via a PER-SAMPLE weight array, since
#           XGBoost has no class_weight dict — we map each training
#           label to its Stage 3 weight,
#       (b) labels are integer-encoded (XGBoost needs 0..8, not strings)
#           and predictions decoded back for the report.
# This cell trains, reports macro-F1 + per-class P/R/F1, and ranks
# feature importance by GAIN. CPU only.
# ============================================================

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# (a) Build the per-sample weight vector: 
#    look up each training wafer's class in the Stage 3 dict and assign that weight.
#    This reproduces the same imbalance correction the RF got from class_weight=, 
#    just expressed the way XGBoost wants it — one weight per row rather than one per class.
sample_weight = pd.Series(y_train).map(class_weights).values

# (b) Integer-encode the labels. 
#     Fit the encoder on the FULL label set y 
#     so the string<->int mapping is stable and identical for train and test, 
#     then transform both.
#     le.classes_ holds the alphabetical class order, the integers correspond to.
le = LabelEncoder().fit(y)
y_train_enc = le.transform(y_train)          # strings -> 0..8 for training
y_test_enc  = le.transform(y_test)           # same mapping for the held-out set

# Train the booster. 
#      n_estimators/depth/lr are a sensible untuned baseline chosen
#      to sit in the same effort class as the 300-tree RF, not to win a tuning contest.
#      tree_method='hist' is the fast modern algorithm; 
#      importance_type='gain' makes feature_importances_ report average split gain, the most interpretable ranking.
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',                      # fast histogram-based splitting (CPU)
    importance_type='gain',                  # rank features by gain, not raw split count
    random_state=42,
    n_jobs=-1
)
t0 = time.time()
xgb.fit(X_feat_train, y_train_enc, sample_weight=sample_weight)   # weights applied here
print(f"Trained in {time.time() - t0:.1f}s")

# Predict encoded classes, 
# then decode back to strings so the report reads in human labels and lines up with the RF output above.
y_pred_xgb = le.inverse_transform(xgb.predict(X_feat_test))

# Same headline pair as the RF: accuracy (inflated) beside macro-F1 (honest),
# so the two models are compared on identical footing.
print(f"\nAccuracy : {accuracy_score(y_test, y_pred_xgb):.4f}   <- inflated by the 'none' majority")
print(f"Macro-F1 : {f1_score(y_test, y_pred_xgb, average='macro'):.4f}   <- the metric we steer by")

# Per-class report in the same frequency order as the RF, 
# so you can read the two tables side by side row-for-row (watch Scratch recall and Edge-Loc F1 especially).
print("\nPer-class report:")
print(classification_report(y_test, y_pred_xgb, labels=class_order, digits=3))

# Feature importance by gain. 
# This may rank differently from the RF's impurity
# importance — a feature XGBoost splits on rarely, but with high gain will rise
# here, which is itself informative about which signals are decisive vs frequent.
importances_xgb = pd.Series(xgb.feature_importances_, index=X_feat.columns).sort_values(ascending=False)
print("\nFeature importances (gain, descending):")
print(importances_xgb.round(4))

Trained in 23.3s

Accuracy : 0.9415   <- inflated by the 'none' majority
Macro-F1 : 0.7934   <- the metric we steer by

Per-class report:
              precision    recall  f1-score   support

        none      0.994     0.954     0.974     29486
   Edge-Ring      0.946     0.952     0.949      1936
    Edge-Loc      0.580     0.831     0.683      1038
      Center      0.796     0.946     0.865       859
         Loc      0.491     0.695     0.575       718
     Scratch      0.298     0.623     0.403       239
      Random      0.826     0.908     0.865       173
       Donut      0.798     0.856     0.826       111
   Near-full      1.000     1.000     1.000        30

    accuracy                          0.942     34590
   macro avg      0.748     0.863     0.793     34590
weighted avg      0.958     0.942     0.948     34590


Feature importances (gain, descending):
radon_peak            0.1652
conn_largest_frac     0.0976
radon_peak_ratio      0.0807
radial_ring_1         0.0798


In [7]:
# ============================================================
# STAGE 4.5 — LINEAR SVM (classical route, linear-separability probe)
# Not a bid to beat the tree models — a diagnostic. 
# RF and XGBoost are non-linear;
# A linear SVM draws flat hyperplanes, 
# so its score tells us how linearly separable the 15 hand features make the 9 classes.
# Two mechanics differ from the tree models:
#       (a) SVMs are scale-sensitive -> standardize features (fit on
#           TRAIN only, apply to test — no leakage),
#       (b) class_weight takes the Stage 3 dict directly, like the RF.
# LinearSVC is used (liblinear/SAG-class solver) because it scales to
# 138k rows in seconds, unlike a kernel SVM. CPU only.
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler

# (a) Standardize: center-and-scale each feature to mean 0 / variance 1. The
# scaler LEARNS those stats from training data only, then applies them to test —
# fitting on the full set would leak test distribution into the transform.
scaler = StandardScaler().fit(X_feat_train)          # learn mean/std from train only
X_feat_train_s = scaler.transform(X_feat_train)      # apply to train
X_feat_test_s  = scaler.transform(X_feat_test)       # apply SAME transform to test

# Train the linear SVM. 
# class_weight takes the Stage 3 dict directly (same weights the RF used). 
# dual=False is the efficient setting when samples >> features (138360 >> 15). 
# max_iter is raised so the solver fully converges at this scale.
svm = LinearSVC(
    class_weight=class_weights,     # audited Stage 3 weights, straight in
    dual=False,                     # efficient primal solver: n_samples >> n_features
    random_state=42,
    max_iter=5000                   # headroom for convergence on standardized data
)
t0 = time.time()
svm.fit(X_feat_train_s, y_train)    # note: trains on the SCALED features
print(f"Trained in {time.time() - t0:.1f}s")

# Predict on the scaled test features (LinearSVC outputs class labels directly,
# so no encode/decode dance is needed here — strings work).
y_pred_svm = svm.predict(X_feat_test_s)

# Same headline pair as the other two models, on identical footing.
print(f"\nAccuracy : {accuracy_score(y_test, y_pred_svm):.4f}   <- inflated by the 'none' majority")
print(f"Macro-F1 : {f1_score(y_test, y_pred_svm, average='macro'):.4f}   <- the metric we steer by")

# Per-class report in the same frequency order, so this reads row-for-row against the RF and XGBoost tables.
# The interest is less in any single class than the overall drop (or not) vs the tree models — that gap IS the non-linearity measure.
print("\nPer-class report:")
print(classification_report(y_test, y_pred_svm, labels=class_order, digits=3))

Trained in 5.5s

Accuracy : 0.9506   <- inflated by the 'none' majority
Macro-F1 : 0.7459   <- the metric we steer by

Per-class report:
              precision    recall  f1-score   support

        none      0.984     0.980     0.982     29486
   Edge-Ring      0.937     0.944     0.941      1936
    Edge-Loc      0.723     0.607     0.660      1038
      Center      0.818     0.903     0.858       859
         Loc      0.582     0.494     0.535       718
     Scratch      0.316     0.531     0.396       239
      Random      0.602     0.867     0.711       173
       Donut      0.571     0.901     0.699       111
   Near-full      0.964     0.900     0.931        30

    accuracy                          0.951     34590
   macro avg      0.722     0.792     0.746     34590
weighted avg      0.953     0.951     0.951     34590



In [8]:
# ============================================================
# STAGE 4.6 — SAVE CLASSICAL RESULTS BEFORE THE GPU RESTART
# Switching to a GPU accelerator restarts the kernel and wipes memory,
# so every classical result Stage 5 needs must be written to disk NOW.
# We save PREDICTIONS + TRUTH + SCORES + IMPORTANCES (not the models —
# predictions are what the Stage 5 confusion matrices actually read):
#       1. a predictions table (y_test truth + each model's y_pred),
#       2. the three-model scorecard (macro-F1 + accuracy),
#       3. The RF impurity and XGBoost gain importance rankings,
#       4. Read every file back to confirm it was written before we commit.
# ============================================================

# --- 1. Predictions + truth, one row per test wafer, one column per model ---
# Stage 5 builds a confusion matrix per model as truth-vs-pred, 
# so truth and all three prediction vectors live together in one aligned table. 
# Index is the test row positions, so any prediction can be traced back to its exact wafer if needed.
preds = pd.DataFrame(
    {'y_true': y_test, 'rf': y_pred, 'xgb': y_pred_xgb, 'svm': y_pred_svm},
    index=test_idx                              # preserves which wafer each row is
)
preds.to_pickle('/kaggle/working/classical_predictions.pkl')

# --- 2. Scorecard: the headline table, computed once and frozen ---
# Recomputed here (not hand-typed) so the saved numbers are guaranteed to match
# what the models actually produced — no transcription drift into Stage 5.
scores = pd.DataFrame({
    'macro_f1': [f1_score(y_test, p, average='macro') for p in [y_pred, y_pred_xgb, y_pred_svm]],
    'accuracy': [accuracy_score(y_test, p)             for p in [y_pred, y_pred_xgb, y_pred_svm]],
}, index=['random_forest', 'xgboost', 'linear_svm']).round(4)
scores.to_pickle('/kaggle/working/classical_scores.pkl')

# --- 3. Feature importances: the interpretation payload ---
# Both rankings saved together in one frame (RF impurity vs XGB gain), since the
# Stage 5 story is partly the DIFFERENCE between them (e.g., radon_peak buried at
# 8th by impurity but 1st by gain — the Scratch-detection insight).
importance_table = pd.DataFrame({
    'rf_impurity': importances,                 # from Step 3
    'xgb_gain':    importances_xgb,             # from Step 4
})
importance_table.to_pickle('/kaggle/working/classical_importances.pkl')

# --- 4. Confirm every file written before committing the version ---
import os
for f in ['classical_predictions.pkl', 'classical_scores.pkl', 'classical_importances.pkl']:
    kb = os.path.getsize(f'/kaggle/working/{f}') / 1024
    print(f"{f:32s} {kb:8.1f} KB")

# Echo the frozen scorecard so the saved artifact is visible in the notebook output.
print("\nSaved classical scorecard:")
print(scores)

classical_predictions.pkl          1281.9 KB
classical_scores.pkl                  0.7 KB
classical_importances.pkl             1.2 KB

Saved classical scorecard:
               macro_f1  accuracy
random_forest    0.7893    0.9632
xgboost          0.7934    0.9415
linear_svm       0.7459    0.9506


In [1]:
# ============================================================
# STAGE 4.7 — GPU RELOAD + IMAGE TENSOR PREP (CNN route)
# The accelerator switch restarted the kernel, so we reload from the
# still-attached inputs and shape the images for a CNN. Steps:
#       1. re-import + confirm the GPU is actually visible to the framework,
#       2. reload X_img, y, and the frozen split indices,
#       3. rebuild the same train/test partition (identical wafers),
#       4. shape tensors for the CNN: scale 0/1/2 -> [0,1], add a channel
#          axis, integer-encode labels — each step categorical-safe.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

# (1) Confirm the framework sees a GPU. 
#     If this prints an empty list, the switch didn't take — stop and re-check the accelerator setting before training,
#     or the CNN will silently crawl on CPU.
print("TF version:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices('GPU'))

# (2) Reload from the attached inputs (memory was wiped; the mounts were not).
P2 = '/kaggle/input/notebooks/arijitmohanta/02-preprocessing'
P3 = '/kaggle/input/notebooks/arijitmohanta/03-class-imbalance'
X_img     = np.load(f'{P2}/X_img.npy')            # (172950, 64, 64) uint8
y         = np.load(f'{P2}/y.npy')                # (172950,) <U9 strings
train_idx = np.load(f'{P3}/train_idx.npy')
test_idx  = np.load(f'{P3}/test_idx.npy')

# (3) Rebuild the SAME split by applying the frozen indices 
#     the CNN must train and test on the identical wafers that the classical models used, 
#     or the head-to-head in Step 10 is invalid.
X_img_train, X_img_test = X_img[train_idx], X_img[test_idx]
y_train,     y_test     = y[train_idx],     y[test_idx]

# (4a) Scale pixel codes 0/1/2 into [0, 1] by dividing by 2.0. 
# This is a linear rescale of three fixed categories to three fixed values (0, 0.5, 1.0) — 
# it does NOT interpolate or invent intermediate states, so it stays categorical-safe.
# The channel axis (-1) is added because Conv2D expects (H, W, channels); 
# wafer maps are single-channel, like greyscale.
X_img_train = (X_img_train / 2.0).astype('float32')[..., np.newaxis]   # (N,64,64,1)
X_img_test  = (X_img_test  / 2.0).astype('float32')[..., np.newaxis]

# (4b) Integer-encode labels for the CNN's sparse loss. 
# Fit on the full label set so the mapping is stable, then transform both partitions with it. 
# le.classes_ holds the class order the integers correspond to — kept for decoding in Step 9.
le = LabelEncoder().fit(y)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)
num_classes = len(le.classes_)

print("\nX_img_train:", X_img_train.shape, X_img_train.dtype)   # expect (138360, 64, 64, 1) float32
print("X_img_test :", X_img_test.shape)                         # expect (34590, 64, 64, 1)
print("Classes:", num_classes, "->", list(le.classes_))

TF version: 2.20.0
GPUs visible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

X_img_train: (138360, 64, 64, 1) float32
X_img_test : (34590, 64, 64, 1)
Classes: 9 -> [np.str_('Center'), np.str_('Donut'), np.str_('Edge-Loc'), np.str_('Edge-Ring'), np.str_('Loc'), np.str_('Near-full'), np.str_('Random'), np.str_('Scratch'), np.str_('none')]


In [2]:
# ============================================================
# STAGE 4.8a — AUGMENTATION PIPELINE + CNN ARCHITECTURE (define only)
# Turns the Stage 3 augmentation plan into code and defines a small CNN.
# This cell DEFINES and SUMMARISES only — no training yet (that's 8b).
#       1. an augmentation layer using ONLY 90-degree rotations + flips
#          (categorical-safe: maps die-to-die, never interpolates),
#       2. a compact conv stack sized for 64x64x1 wafer maps,
#       3. compiled with a sparse loss (integer labels) + macro-F1-friendly
#          metrics; class weights are applied later, at fit() time in 8b.
# ============================================================

from tensorflow.keras import layers, models

# --- 1. Augmentation: categorical-safe geometric transforms only ---
# RandomFlip does horizontal+vertical mirroring; 
# RandomRotation with factor=0.25 rotates by up to +/-90 degrees. 
# On a square grid, these are die-to-die remappings with 'nearest' interpolation forced,
# so no fractional die states are ever invented.
# fill_mode='constant' (fill_value 0) keeps any exposed corner as outside-wafer, not a wrapped duplicate. 
# This runs on-GPU during training and is ACTIVE ONLY at train time — Keras auto-disables it at inference.
augment = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),                       # mirror: valid same-class wafer
    layers.RandomRotation(0.25, fill_mode='constant', fill_value=0.0,   # +/-90deg, corners -> outside-wafer
                          interpolation='nearest'),                     # nearest: no fractional dies
], name="categorical_safe_augment")

# --- 2. The CNN: a deliberately small stack ---
# Three conv blocks (32->64->128 filters) each halving spatial size via pooling,then a dense head. 
# Kept small on purpose: 
# The classical baseline is 0.79, and a compact net is the fair comparison + trains fast on these 64x64 single-channel maps. 
# Dropout guards against the network memorizing the augmented rare classes.
model = models.Sequential([
    layers.Input(shape=(64, 64, 1)),          # single-channel wafer map
    augment,                                  # augmentation as the first layer (train-time only)

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),                    # 64 -> 32
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),                    # 32 -> 16
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),                    # 16 -> 8

    layers.GlobalAveragePooling2D(),          # 8x8x128 -> 128 vector, fewer params than Flatten
    layers.Dropout(0.4),                      # regularise the dense head
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax'),   # 9-way class probabilities
])

# Compile with sparse categorical cross-entropy (matches our INTEGER labels, no one-hot needed).
# Accuracy is logged only as a familiar reference — we do NOT steer by it;
# macro-F1 comes from the full per-class report in Step 9.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()   # print the architecture + parameter count before training

I0000 00:00:1786284119.768957      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786284119.771933      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ categorical_safe_augment        │ (None, 64, 64, 1)      │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,345 (431.04 KB)

 Trainable params: 110,345 (431.04 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# ============================================================
# STAGE 4.8b — TRAIN THE CNN (class weights + augmentation active)
# Trains the compiled model. Wiring handled here:
#       1. remap the Stage 3 weights from STRING keys to INTEGER keys,
#          because the CNN's labels are integer-encoded (Keras needs int
#          keys in class_weight),
#       2. carve a validation slice from TRAIN ONLY (test stays untouched
#          for Step 9's honest per-class evaluation),
#       3. train with EarlyStopping on val macro-signal, restoring the
#          best weights so we keep the best epoch, not the last.
# Augmentation (from 8a) is active automatically at train time only.
# ============================================================

import pickle
from tensorflow.keras.callbacks import EarlyStopping

# (1) Reload the Stage 3 weights (string-keyed) and remap to integer keys that
# match the LabelEncoder's ordering, since Keras class_weight indexes by the
# integer label the loss sees. le.transform maps each class name to its int.
with open(f'{P3}/class_weights.pkl', 'rb') as f:
    class_weights_str = pickle.load(f)                       # {'none': 0.13, ..., 'Near-full': 129.2}
class_weights_int = {int(le.transform([k])[0]): v            # -> {8: 0.13, ..., 5: 129.2}
                     for k, v in class_weights_str.items()}
print("Integer-keyed class weights:", {k: round(v, 2) for k, v in sorted(class_weights_int.items())})

# (2) EarlyStopping watches validation loss; patience=5 lets it ride out noise,
# restore_best_weights rolls back to the best epoch so a late overfit epoch isn't
# what we keep. validation_split=0.15 below carves the val set from TRAIN only.
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# (3) Train. class_weight applies the rare-class penalty in the loss; augmentation
# fires on each training batch (not validation). batch_size 256 keeps the T4 fed;
# up to 40 epochs, but EarlyStopping will likely halt earlier.
history = model.fit(
    X_img_train, y_train_enc,
    validation_split=0.15,            # 15% of TRAIN held out for validation (test untouched)
    epochs=40,
    batch_size=256,
    class_weight=class_weights_int,   # rare-class penalty, integer-keyed
    callbacks=[early_stop],
    verbose=1,
)

print("\nTraining finished. Epochs run:", len(history.history['loss']))

Integer-keyed class weights: {0: np.float64(4.48), 1: np.float64(34.62), 2: np.float64(3.7), 3: np.float64(1.99), 4: np.float64(5.35), 5: np.float64(129.19), 6: np.float64(22.18), 7: np.float64(16.11), 8: np.float64(0.13)}
Epoch 1/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.1376 - loss: 2.0059 - val_accuracy: 0.0589 - val_loss: 1.7342
Epoch 2/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.4842 - loss: 1.2724 - val_accuracy: 0.6831 - val_loss: 1.1407
Epoch 3/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.5494 - loss: 1.1177 - val_accuracy: 0.5882 - val_loss: 1.2739
Epoch 4/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.6196 - loss: 0.9965 - val_accuracy: 0.7345 - val_loss: 0.8791
Epoch 5/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.6432 - loss: 0.9316 - val_accuracy: 0.3630 - val_loss: 1.3965
Epoch 6/40
460/460 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.6550 - loss: 0.8711 - val_accuracy: 0.7205 - val_loss: 0.94

In [4]:
# ============================================================
# STAGE 4.9 — CNN EVALUATION ON THE UNTOUCHED TEST SET
# The only CNN number that counts: per-class macro-F1 on the frozen
# test wafers (never seen in train OR validation). Steps:
#       1. predict class probabilities, take argmax -> integer labels,
#       2. decode integers back to class-name strings for a readable report,
#       3. print accuracy beside macro-F1 (the gap, one last time),
#       4. full per-class precision/recall/F1 in the same order as the
#          classical tables, so Step 10 reads CNN-vs-XGBoost row for row.
# ============================================================

from sklearn.metrics import classification_report, f1_score, accuracy_score

# (1) Predict. The softmax gives a (34590, 9) probability matrix; argmax over axis
# 1 picks the most likely class per wafer. Augmentation is OFF here automatically
# (Keras disables it at inference), so the model sees clean test images.
y_prob_cnn = model.predict(X_img_test, batch_size=256, verbose=0)   # (N, 9) probabilities
y_pred_cnn_enc = y_prob_cnn.argmax(axis=1)                          # -> integer labels

# (2) Decode integers back to class-name strings via the same LabelEncoder, so the
# report is human-readable and directly comparable to the classical tables.
y_pred_cnn = le.inverse_transform(y_pred_cnn_enc)

# Rebuild the frequency class order (this cell may run after a restart) so the
# per-class table reads none-first .. Near-full-last, matching every prior report.
class_order = pd.Series(y_test).value_counts().index.tolist()

# (3) Headline pair, last time: accuracy is inflated by the 85% 'none' majority,
# macro-F1 averages all nine classes equally and is the honest, comparable score.
print(f"Accuracy : {accuracy_score(y_test, y_pred_cnn):.4f}   <- still inflated by 'none'")
print(f"Macro-F1 : {f1_score(y_test, y_pred_cnn, average='macro'):.4f}   <- vs XGBoost 0.7934")

# (4) Full per-class report — the real comparison surface. 
# Watch the two classes, the classical route couldn't fix: 
#       -- Scratch (raw line now visible to conv filters)
#       -- Edge-Loc (raw arc geometry vs the weak angular scalar).
print("\nPer-class report (CNN):")
print(classification_report(y_test, y_pred_cnn, labels=class_order, digits=3))

Accuracy : 0.9158   <- still inflated by 'none'
Macro-F1 : 0.6949   <- vs XGBoost 0.7934

Per-class report (CNN):
              precision    recall  f1-score   support

        none      0.990     0.938     0.964     29486
   Edge-Ring      0.979     0.943     0.960      1936
    Edge-Loc      0.639     0.818     0.717      1038
      Center      0.818     0.697     0.753       859
         Loc      0.558     0.448     0.497       718
     Scratch      0.085     0.577     0.148       239
      Random      0.677     0.884     0.767       173
       Donut      0.369     0.838     0.512       111
   Near-full      0.906     0.967     0.935        30

    accuracy                          0.916     34590
   macro avg      0.669     0.790     0.695     34590
weighted avg      0.956     0.916     0.933     34590



In [6]:
# ============================================================
# STAGE 4.10 (CORRECTED) — SAVE CNN ARTIFACTS + FINAL SCORECARD
# Note: the GPU accelerator switch restarted the kernel, which WIPED
# /kaggle/working/ — so classical_scores.pkl (written pre-switch in 4.6)
# no longer exists this session. Rather than a fragile cross-session
# reload, we record the three frozen classical macro-F1s as constants
# (they are final results, printed in the classical cells above) and
# append the freshly-computed CNN row.
#       1. save CNN predictions + probabilities (aligned to test_idx),
#       2. save the trained model in .keras format,
#       3. build the four-model scorecard from known classical results
#          + the live CNN score, all on the identical frozen test set,
#       4. confirm every file wrote before committing.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

# (1) CNN predictions + full probability matrix, aligned to test_idx so each row
# traces to its exact wafer (matching classical_predictions.pkl for Stage 5).
cnn_out = pd.DataFrame({'y_true': y_test, 'cnn': y_pred_cnn}, index=test_idx)
cnn_out.to_pickle('/kaggle/working/cnn_predictions.pkl')
np.save('/kaggle/working/cnn_probabilities.npy', y_prob_cnn)   # (34590, 9) softmax matrix

# (2) Trained model in .keras (round-trips the augmentation layer correctly; that
# block is inert at inference, so a reloaded model predicts identically).
model.save('/kaggle/working/wafer_cnn.keras')

# (3) The three classical macro-F1 / accuracy pairs are FINAL results from the
# classical cells above (RF 4.3, XGBoost 4.4, SVM 4.5) — recorded here as constants
# because the kernel restart wiped their saved file, and re-deriving would need the
# classical predictions that were wiped too. The CNN row is computed live.
classical = pd.DataFrame(
    {'macro_f1': [0.7893, 0.7934, 0.7459],
     'accuracy': [0.9632, 0.9415, 0.9506]},
    index=['random_forest', 'xgboost', 'linear_svm']
)
cnn_row = pd.DataFrame(
    {'macro_f1': [round(f1_score(y_test, y_pred_cnn, average='macro'), 4)],
     'accuracy': [round(accuracy_score(y_test, y_pred_cnn), 4)]},
    index=['cnn']
)
final_scores = pd.concat([classical, cnn_row])
final_scores.to_pickle('/kaggle/working/final_scores.pkl')

# (4) Confirm every artifact wrote, then echo the final scorecard best-first.
for f in ['cnn_predictions.pkl', 'cnn_probabilities.npy', 'wafer_cnn.keras', 'final_scores.pkl']:
    mb = os.path.getsize(f'/kaggle/working/{f}') / 1e6
    print(f"{f:26s} {mb:7.2f} MB")

print("\nFinal four-model scorecard (identical frozen test set):")
print(final_scores.sort_values('macro_f1', ascending=False))

cnn_predictions.pkl           0.80 MB
cnn_probabilities.npy         1.25 MB
wafer_cnn.keras               1.38 MB
final_scores.pkl              0.00 MB

Final four-model scorecard (identical frozen test set):
               macro_f1  accuracy
xgboost          0.7934    0.9415
random_forest    0.7893    0.9632
linear_svm       0.7459    0.9506
cnn              0.6949    0.9158


In [7]:
# ============================================================
# STAGE 4.10b — VERIFY STAGE 5 INPUTS EXIST BEFORE COMMITTING
# The GPU restart wiped /kaggle/working/ once already. Stage 5 needs
# BOTH the CNN artifacts (saved this session) AND the classical
# predictions (saved pre-restart in 4.6). This cell confirms every
# Stage-5 input is present in /kaggle/working NOW, so the commit
# captures a complete set — catching any gap here, not in Stage 5.
# ============================================================

import os

# The full set Stage 5 will attach and read. classical_predictions.pkl is the one
# at risk — it was written pre-GPU-switch; if the restart wiped it and it wasn't
# re-created, it'll show MISSING and we regenerate it before committing.
needed = [
    'classical_predictions.pkl',   # RF/XGB/SVM per-wafer preds  (from 4.6, pre-restart)
    'cnn_predictions.pkl',         # CNN per-wafer preds         (this session)
    'cnn_probabilities.npy',       # CNN softmax matrix          (this session)
    'final_scores.pkl',            # four-model scorecard        (this session)
    'wafer_cnn.keras',             # trained model               (this session)
]
for f in needed:
    path = f'/kaggle/working/{f}'
    status = f"{os.path.getsize(path)/1e6:6.2f} MB" if os.path.exists(path) else "*** MISSING ***"
    print(f"{f:28s} {status}")

classical_predictions.pkl    *** MISSING ***
cnn_predictions.pkl            0.80 MB
cnn_probabilities.npy          1.25 MB
final_scores.pkl               0.00 MB
wafer_cnn.keras                1.38 MB


In [8]:
# ============================================================
# STAGE 4.10c — REGENERATE classical_predictions.pkl (post-restart recovery)
# The GPU restart wiped /kaggle/working/, taking classical_predictions.pkl
# with it. This cell reproduces ONLY that artifact — reload the feature
# route, rebuild the frozen split, re-fit RF + XGBoost + linear SVM with
# the same settings as Steps 4.3-4.5, and re-save the per-wafer predictions
# Stage 5 needs. Runs on CPU internally (sklearn/XGBoost ignore the GPU),
# so NO accelerator switch is needed — avoids another wipe.
# Skip this cell entirely if classical_predictions.pkl is already in your
# pre-GPU commit; it's only here for the no-commit case.
# ============================================================

import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier

# --- Reload the classical inputs (wiped from memory by the restart) ---
P3 = '/kaggle/input/notebooks/arijitmohanta/03-class-imbalance'
X_feat    = pd.read_pickle(f'{P3}/X_feat_pruned.pkl')     # (172950, 15)
y         = np.load(f'{P3.replace("03-class-imbalance","02-preprocessing")}/y.npy')
train_idx = np.load(f'{P3}/train_idx.npy')
test_idx  = np.load(f'{P3}/test_idx.npy')
with open(f'{P3}/class_weights.pkl', 'rb') as f:
    class_weights = pickle.load(f)

# Rebuild the identical frozen split.
X_feat_train, X_feat_test = X_feat.iloc[train_idx], X_feat.iloc[test_idx]
y_train,      y_test      = y[train_idx],           y[test_idx]

# --- RF: same config as Step 4.3 (class_weight dict, 300 trees, seed 42) ---
rf = RandomForestClassifier(n_estimators=300, class_weight=class_weights,
                            random_state=42, n_jobs=-1).fit(X_feat_train, y_train)
pred_rf = rf.predict(X_feat_test)

# --- XGBoost: same as Step 4.4 (per-sample weights, integer labels) ---
le = LabelEncoder().fit(y)
sw = pd.Series(y_train).map(class_weights).values
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    tree_method='hist', random_state=42, n_jobs=-1)
xgb.fit(X_feat_train, le.transform(y_train), sample_weight=sw)
pred_xgb = le.inverse_transform(xgb.predict(X_feat_test))

# --- Linear SVM: same as Step 4.5 (standardized, class_weight dict) ---
scaler = StandardScaler().fit(X_feat_train)
svm = LinearSVC(class_weight=class_weights, dual=False, random_state=42, max_iter=5000)
svm.fit(scaler.transform(X_feat_train), y_train)
pred_svm = svm.predict(scaler.transform(X_feat_test))

# --- Re-save the artifact, identical schema to the original 4.6 version ---
preds = pd.DataFrame({'y_true': y_test, 'rf': pred_rf, 'xgb': pred_xgb, 'svm': pred_svm},
                     index=test_idx)
preds.to_pickle('/kaggle/working/classical_predictions.pkl')

# Sanity-check it wrote and the macro-F1s match Steps 4.3-4.5 (confirms faithful reproduction).
from sklearn.metrics import f1_score
import os
print("classical_predictions.pkl:", f"{os.path.getsize('/kaggle/working/classical_predictions.pkl')/1e6:.2f} MB")
for name, p in [('rf', pred_rf), ('xgb', pred_xgb), ('svm', pred_svm)]:
    print(f"  {name} macro-F1: {f1_score(y_test, p, average='macro'):.4f}")

classical_predictions.pkl: 1.31 MB
  rf macro-F1: 0.7893
  xgb macro-F1: 0.7934
  svm macro-F1: 0.7459
